# 01 - Exploratory data analysis

Class balance, duplicates, amount and time patterns, and which PCA components separate fraud in `creditcard.csv`.
Run from the `notebooks/` folder after downloading the data (see `data/README.md`).

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.data import TARGET, V_COLS

raw = pd.read_csv("../data/creditcard.csv")
raw.shape

## Class balance and duplicates

In [ ]:
print(raw[TARGET].value_counts())
print(f"fraud rate: {raw[TARGET].mean():.4%}")
dups = raw.duplicated()
print("exact duplicate rows:", int(dups.sum()), "| of which fraud:", int(raw.loc[dups, TARGET].sum()))
df = raw.drop_duplicates().reset_index(drop=True)
print(f"after dropping duplicates: {len(df):,} rows, {int(df[TARGET].sum())} fraud ({df[TARGET].mean():.4%})")
print("missing values:", int(raw.isna().sum().sum()))

## Amount by class

In [ ]:
print(df.groupby(TARGET)["Amount"].describe(percentiles=[.25, .5, .75, .99]).round(2))
print()
print("fraud transactions with Amount < 2:", int(((df[TARGET] == 1) & (df["Amount"] < 2)).sum()),
      "of", int(df[TARGET].sum()))

## Time patterns

`Time` counts seconds from the first transaction, so the hour below is relative to the start of the capture window, not a clock time. Fraud counts per hour are small, so read the shape cautiously.

In [ ]:
hour = ((df["Time"] % 86400) // 3600).astype(int)
by_hour = df.groupby(hour)[TARGET].agg(transactions="size", fraud="sum")
by_hour["fraud_rate_pct"] = (100 * by_hour["fraud"] / by_hour["transactions"]).round(3)
by_hour.T

## Which PCA components separate fraud?

Univariate ROC-AUC of each component on its own (0.5 means no separation). Values far from 0.5 in either direction are informative.

In [ ]:
auc = pd.Series({c: roc_auc_score(df[TARGET], df[c]) for c in V_COLS})
sep = (auc - 0.5).abs().sort_values(ascending=False)
top = sep.head(10).index
print(auc[top].round(3))
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top[::-1], auc[top][::-1] - 0.5, left=0.5, color="#4C78A8")
ax.axvline(0.5, color="k", lw=1)
ax.set_title("Univariate ROC-AUC of the 10 most separating components")
ax.set_xlabel("ROC-AUC")
plt.tight_layout()